# Gradient Descent — Building IntuitionThis notebook builds intuition for **gradient descent**, the optimization workhorse behind nearly every machine learning algorithm covered in this course (linear regression by GD, logistic regression, perceptron, neural networks).**The idea:** to minimize a function $f(\theta)$, repeatedly step in the direction of steepest descent:$$\theta_{t+1} = \theta_t - \eta \cdot \nabla f(\theta_t)$$We'll demonstrate this on:1. A simple convex function $f(x) = (x-3)^2$ — visualize the trajectory2. **Linear regression by gradient descent** on synthetic data — compare to the closed-form solution

In [ ]:
import numpy as npimport matplotlib.pyplot as pltimport seaborn as snssns.set_style("whitegrid")np.random.seed(42)

## 1. GD on a 1-D convex function

In [ ]:
def f(x):  return (x - 3) ** 2def grad(x): return 2 * (x - 3)# Run GD from x=10xs = [10.0]lr = 0.1for _ in range(30):    xs.append(xs[-1] - lr * grad(xs[-1]))xs = np.array(xs)xx = np.linspace(-2, 11, 300)fig, ax = plt.subplots(1, 2, figsize=(12, 4))ax[0].plot(xx, f(xx), "b-", label="f(x) = (x-3)^2")ax[0].scatter(xs, f(xs), c=np.arange(len(xs)), cmap="viridis", s=40)ax[0].set_xlabel("x"); ax[0].set_ylabel("f(x)")ax[0].set_title("GD trajectory (color = step #)")ax[0].legend()ax[1].plot(f(xs), marker="o")ax[1].set_xlabel("Iteration"); ax[1].set_ylabel("f(x)")ax[1].set_title("Loss over iterations")plt.tight_layout(); plt.show()print(f"Converged to x = {xs[-1]:.4f} (true min: x=3)")

## 2. Effect of learning rate

In [ ]:
def gd(lr, n=30, x0=10.0):    xs = [x0]    for _ in range(n):        xs.append(xs[-1] - lr * grad(xs[-1]))    return np.array(xs)plt.figure(figsize=(10, 4))for lr in [0.01, 0.1, 0.5, 0.9, 1.01]:    plt.plot(f(gd(lr)), label=f"lr={lr}", marker="o", ms=4)plt.yscale("log")plt.xlabel("Iteration"); plt.ylabel("f(x)  (log scale)")plt.title("Loss curves at different learning rates")plt.legend(); plt.tight_layout(); plt.show()

With **lr=1.01** GD diverges — the step size overshoots the minimum each iteration. **lr=0.5** still converges but oscillates. Smaller lrs are slower but stable. This is one of the most important hyperparameter trade-offs in all of ML.

## 3. Linear regression by gradient descent

In [ ]:
# Synthetic data: y = 2x + 1 + noiseN = 100X = np.linspace(0, 10, N).reshape(-1, 1)y = 2 * X.ravel() + 1 + np.random.normal(scale=1.5, size=N)# Closed-form (normal equations) solutionX_aug = np.hstack([np.ones((N, 1)), X])theta_closed = np.linalg.lstsq(X_aug, y, rcond=None)[0]print(f"Closed-form: intercept={theta_closed[0]:.4f}, slope={theta_closed[1]:.4f}")

In [ ]:
# Gradient descent from scratchdef lr_loss(theta, X, y):    return ((X @ theta - y) ** 2).mean()def lr_grad(theta, X, y):    return 2 * X.T @ (X @ theta - y) / len(y)theta = np.zeros(2)lr = 0.01losses = []for _ in range(2000):    theta = theta - lr * lr_grad(theta, X_aug, y)    losses.append(lr_loss(theta, X_aug, y))print(f"GD result:    intercept={theta[0]:.4f}, slope={theta[1]:.4f}")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4))ax[0].plot(losses)ax[0].set_xlabel("Iteration"); ax[0].set_ylabel("MSE loss")ax[0].set_title("Linear regression — GD loss curve")ax[1].scatter(X, y, alpha=0.5, label="data")xs = np.array([0, 10])ax[1].plot(xs, theta[0] + theta[1] * xs, "r-", lw=2, label="GD fit")ax[1].set_xlabel("x"); ax[1].set_ylabel("y")ax[1].legend(); ax[1].set_title("Final fitted line")plt.tight_layout(); plt.show()

## Takeaways- **Gradient descent** is the universal optimization recipe in ML. Pick a loss, compute its gradient, step in the negative gradient direction.- The **learning rate** is critical: too big → divergence; too small → painfully slow.- For linear regression, GD converges to the same solution as the closed-form normal equations — but GD scales to problems where the closed form is intractable (millions of features, neural networks).- Variants you'll encounter later: **stochastic GD** (one sample per step), **mini-batch GD** (small batches), **momentum**, **Adam** (adaptive learning rates).